# Setup a FHIR Server in a new InterSystems IRIS Environment

If you have run the [docker-iris-fhir/start.sh](../../docker-iris-fhir/start.sh) script, as detailed so far, these steps have already been run for you. 

However, if you wish to create a new FHIR Server, or create a FHIR server on a clean instance of InterSystems IRIS For Health (e.g. on a cloud deployment) keep reading...

## Pre-requisites

The scripts in this notebook can be run standalone from any other tutorial. 

### Installs 

The only dependency is the `intersystems-irispython` driver which can be installed with: 

```bash
pip install intersystems-irispython
```

This driver accesses remote InterSystems IRIS servers for SQL queries (seen in other tutorials), but can also remotely execute code in the InterSystems IRIS kernel via the Python Native API (used here). 

### Knowledge

To create a FHIR Server on a new InterSystems IRIS for Health deployment you need the following: 

- Server address (IP Address or web address)
- Superserver port (1972 is the default)
- Username and Password

For demo deployments, you can use the user `SuperUser`. If you have not changed the password, the default password is `SYS`. Please note, the password has expired by default, so you will be prompted to change it on first login through the management portal (available at  http://{SERVER}:52773/csp/sys/%25CSP.Portal.Home.zen). 

**If the password has expired, you cannot connect from Python**. 

**Please note, there may be a couple of minutes between the container starting and InterSystems IRIS for Health starting, during which time the connection will fail.**


### Things to define

Along with this information, you also need to define (i.e. create names) for the following: 

- **Namespace name** (the namespace will be created below).
    - A namespace is an organisational unit in IRIS which includes data and code. For a hackathon project you can probably just stick to a single namespace with a simple name.
    - The rest of the tutorial uses `FHIRSERVER`, you can use the same in your deployment.


- **FHIR Server endpoint**
    - This is the address at which your FHIR server will be available.
    - It needs to start with a `/` and be a valid URL path.
    - Again, the tutorial uses `/fhir/r4`, you can do the same here or use something different.





## Code

To create a new FHIR Server, you can use the following: 

In [ ]:
import iris


## Set credentials here  
HOST = "localhost" # could be an IP address, e.g. 192.12.22.142
PORT = 1972 # Default super server port 
USER = "superuser" # Default Username 
PASSWORD = "SYS" # Default password 



def connect(namespace):
    """
    Connect to IRIS with Python Native API connection 
    using the global connection parameters (Defined above)
    """
    conn = iris.connect(HOST, PORT, namespace, USER, PASSWORD)
    return conn, iris.createIRIS(conn)


def create_fhir_server(namespace, endpoint, overwrite=False):
    """
    Create a FHIR server in a specified namespace and a defined endpoint 
    """
    
    conn, irispy = connect("HSLIB")
    try:
        ns_exists = irispy.classMethodBoolean("%SYS.Namespace", "Exists", namespace)
        if not ns_exists:
            irispy.classMethodVoid("HS.Util.Installer.Foundation", "Install", namespace)
    finally:
        conn.close()

    conn, irispy = connect(namespace)
    try:
        exists = irispy.classMethodBoolean("HS.FHIRServer.ServiceInstance", "ExistsByUrl", endpoint)

        if exists and not overwrite:
            print("FHIR server already exists, skipping...")
            return

        if exists and overwrite:
            print("FHIR server exists, overwriting...")
            irispy.classMethodVoid("HS.FHIRServer.Installer", "UninstallInstance", endpoint)

        strategy_class = "HS.FHIRServer.Storage.JsonAdvSQL.InteractionsStrategy"
        metadata = "hl7.fhir.r4.core@4.0.1"

        irispy.classMethodVoid("HS.FHIRServer.Installer", "InstallNamespace")
        irispy.classMethodVoid("HS.FHIRServer.Installer", "InstallInstance", endpoint, strategy_class, metadata)
    finally:
        conn.close()


create_fhir_server("FHIRSERVER", "/fhir/r4")


This code above should take a while to run but will create the FHIR server. 

The address will depend on the web port. If you are using the community edition of InterSystems IRIS For Health (which you probably are), the default web port is `52773`. The FHIR address is as follows: 

```
http://<SERVER>:<WEB PORT><ENDPOINT>
```
This translates to: 

http://123.45.67.890:52773/fhir/r4 

You can check that the FHIR Server is functioning by going to http://123.45.67.890:52773/fhir/r4/metadata, which should download a FHIR Capability statement file if it is functioning. 